# Wine Cultivar Classification — MLOps Pipeline Walkthrough

**Assignment 1 | MLOps**  
End-to-end pipeline: data validation → dual-classifier training → MLflow tracking → model registry → quality gate.

---

## 1. Environment Check

In [1]:
import sys
import mlflow
import sklearn
import pandas as pd
import numpy as np

print(f"Python     : {sys.version.split()[0]}")
print(f"MLflow     : {mlflow.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")

Python     : 3.14.4
MLflow     : 3.16.1
scikit-learn: 1.9.1
pandas     : 3.0.6
numpy      : 2.5.3


## 2. Data Loading and Validation

In [2]:
import sys
sys.path.insert(0, '.')
from src.data import load_and_split

X_train, X_test, y_train, y_test, feature_names = load_and_split()

print(f"\nTrain split : {X_train.shape}")
print(f"Test split  : {X_test.shape}")
print(f"Test ratio  : {len(X_test)/(len(X_train)+len(X_test))*100:.1f}%")
print(f"\nClass distribution (train):")
print(y_train.value_counts())

[data] Validation passed: 178 samples, 13 features, 3 classes.

Train split : (142, 13)
Test split  : (36, 13)
Test ratio  : 20.2%

Class distribution (train):
target
1    57
0    47
2    38
Name: count, dtype: int64


In [3]:
from sklearn.datasets import load_wine
import pandas as pd

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
display(df.iloc[:3, :5])

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium
0,14.23,1.71,2.43,15.6,127.0
1,13.20,1.78,2.14,11.2,100.0
2,13.16,2.36,2.67,18.6,101.0


## 3. Hyperparameter Grids

In [4]:
RF_GRID = [
    {"n_estimators": 50,  "max_depth": 3,    "min_samples_split": 2},
    {"n_estimators": 100, "max_depth": 5,    "min_samples_split": 4},
    {"n_estimators": 200, "max_depth": None, "min_samples_split": 2},
]

GBM_GRID = [
    {"n_estimators": 50,  "learning_rate": 0.10, "max_depth": 2},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 150, "learning_rate": 0.01, "max_depth": 4},
]

print(f"RandomForest configs : {len(RF_GRID)}")
print(f"GradientBoosting configs : {len(GBM_GRID)}")
print(f"Total runs to log  : {len(RF_GRID)+len(GBM_GRID)}")

RandomForest configs : 3
GradientBoosting configs : 3
Total runs to log  : 6


## 4. MLflow Experiment Results

Training was executed via `python src/train.py` (or `make train`).  
Below we query the results directly from the MLflow tracking store.

In [5]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri("sqlite:///mlruns.db")
client = mlflow.tracking.MlflowClient()

exp = client.get_experiment_by_name("Wine-Cultivar-Classification")
runs = client.search_runs(exp.experiment_id, order_by=["metrics.cv_val_f1_macro DESC"])

print(f"Experiment : {exp.name}")
print(f"Total runs : {len(runs)}")

rows = []
for r in runs:
    rows.append({
        "run_id":       r.info.run_id[:8],
        "model_family": r.data.tags.get("model_family", "?"),
        "cv_val_f1":    round(r.data.metrics.get("cv_val_f1_macro", 0), 4),
        "test_f1":      round(r.data.metrics.get("test_f1_macro", 0), 4),
        "test_acc":     round(r.data.metrics.get("test_accuracy", 0), 4),
        "latency_ms":   round(r.data.metrics.get("inference_latency_ms", 0), 2),
    })

df_runs = pd.DataFrame(rows)
display(df_runs)

Experiment : Wine-Cultivar-Classification
Total runs : 7


run_id,model_family,cv_val_f1,test_f1,test_acc,latency_ms
1c193887,RandomForest,0.9789,1.0000,1.0000,168.88
f3af500a,RandomForest,0.9789,1.0000,1.0000,96.17
c3a96653,RandomForest,0.9665,1.0000,1.0000,40.66
517cd4ec,RandomForest,0.9665,1.0000,1.0000,43.03
fd922ce7,GradientBoosting,0.9593,0.9453,0.9444,9.89
e5fb884f,GradientBoosting,0.9114,0.9453,0.9444,8.42
c14c50d9,GradientBoosting,0.8987,0.8907,0.9167,4.73


## 5. Champion Model — Registry

In [6]:
model_name = "WineClassifier"
versions = client.search_model_versions(f"name='{model_name}'")

for v in versions:
    aliases = client.get_model_version_by_alias(model_name, "champion") if v.aliases else None
    print(f"Registered model : {model_name}")
    print(f"Version          : {v.version}")
    print(f"Aliases          : {v.aliases}")
    print(f"Source run_id    : {v.run_id}")

Registered model : WineClassifier
Version          : 1
Aliases          : ['champion']
Source run_id    : f3af500aebe645019da9dd533c6c5a90


## 6. Champion Model — Test Evaluation

In [7]:
from src.evaluate import evaluate_champion
metrics = evaluate_champion()

[evaluate] Loading model: models:/WineClassifier@champion
[data] Validation passed: 178 samples, 13 features, 3 classes.

=== Champion Model — Test Split Evaluation ===
  Macro F1-Score  : 1.0000
  Accuracy        : 1.0000
  Log Loss        : 0.1114
  Latency (batch) : 26.24 ms

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       1.00      1.00      1.00        14
     class_2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

[evaluate] Output schema OK — all predictions are in {0, 1, 2}.


## 7. MLOps Quality Gate Results

In [8]:
F1_THRESHOLD = 0.88
LATENCY_THRESHOLD_MS = 30.0

f1    = metrics["test_f1_macro"]
lat   = metrics["inference_latency_ms"]
preds = set(metrics["predictions"])

print(f"Gate 1 — F1 Threshold (>= {F1_THRESHOLD}):")
print(f"  Champion F1 = {f1:.4f}  {'✓  PASSED' if f1 >= F1_THRESHOLD else '✗  FAILED'}")

print(f"\nGate 2 — Latency (<= {LATENCY_THRESHOLD_MS} ms):")
print(f"  Latency     = {lat:.2f} ms  {'✓  PASSED' if lat <= LATENCY_THRESHOLD_MS else '✗  FAILED'}")

print(f"\nGate 3 — Output Schema ({{0,1,2}} only):")
print(f"  Classes     = {preds}  {'✓  PASSED' if preds.issubset({0,1,2}) else '✗  FAILED'}")

Gate 1 — F1 Threshold (>= 0.88):
  Champion F1 = 1.0000  ✓  PASSED

Gate 2 — Latency (<= 30 ms):
  Latency     = 26.24 ms  ✓  PASSED

Gate 3 — Output Schema ({0,1,2} only):
  Classes     = {0, 1, 2}  ✓  PASSED


## 8. MLflow UI

To open the MLflow tracking UI locally:

```bash
mlflow ui --backend-store-uri sqlite:///mlruns.db --port 5000
```

Then open **http://localhost:5000** in your browser.

You will see:
- **Experiment** `Wine-Cultivar-Classification` with 6+ runs
- Each run with logged parameters, metrics, and model artifacts
- **Model Registry** → `WineClassifier` → version 1 with alias `champion`

## 9. Hyperparameter Search Summary (Table 1 for Report)

In [9]:
print("Table 1: Hyperparameter Search Results")
print("=" * 66)
print(f"{'Family':<18}{'Params':<32}{'CV F1':>7}  {'Test F1':>7}  {'Latency':>9}")
print("-" * 66)

table = [
    ("RandomForest",     "n=100, depth=5, split=4",    0.9789, 1.0000,  96.17, True),
    ("RandomForest",     "n=200, depth=None, split=2", 0.9789, 1.0000, 168.88, False),
    ("RandomForest",     "n=50,  depth=3, split=2",    0.9665, 1.0000,  40.66, False),
    ("GradientBoosting", "n=50,  lr=0.10, depth=2",   0.9593, 0.9453,   9.89, False),
    ("GradientBoosting", "n=100, lr=0.05, depth=3",   0.9114, 0.9453,   8.42, False),
    ("GradientBoosting", "n=150, lr=0.01, depth=4",   0.8987, 0.8907,   4.73, False),
]

for fam, p, cv_f1, t_f1, lat, is_champ in table:
    mark = "  ← CHAMPION" if is_champ else ""
    print(f"{fam:<18}{p:<32}{cv_f1:>7.4f}  {t_f1:>7.4f}  {lat:>7.2f}ms{mark}")

print("=" * 66)
print("Champion selected by: max cv_val_f1_macro | Threshold: F1 >= 0.88")

Table 1: Hyperparameter Search Results
Family            Params                          CV F1   Test F1  Latency
------------------------------------------------------------------
RandomForest      n=100, depth=5, split=4         0.9789  1.0000   96.17ms  ← CHAMPION
RandomForest      n=200, depth=None, split=2      0.9789  1.0000  168.88ms
RandomForest      n=50,  depth=3, split=2         0.9665  1.0000   40.66ms
GradientBoosting  n=50,  lr=0.10, depth=2         0.9593  0.9453    9.89ms
GradientBoosting  n=100, lr=0.05, depth=3         0.9114  0.9453    8.42ms
GradientBoosting  n=150, lr=0.01, depth=4         0.8987  0.8907    4.73ms
Champion selected by: max cv_val_f1_macro | Threshold: F1 >= 0.88


## 10. Run All Tests

In [10]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)

============================= test session starts ==============================
platform linux -- Python 3.14.4, pytest-8.3.5, pluggy-1.6.0
collecting ... collected 17 items

tests/test_data.py::TestLoadAndSplit::test_train_test_shapes PASSED      [  5%]
tests/test_data.py::TestLoadAndSplit::test_split_ratio PASSED            [ 11%]
tests/test_data.py::TestLoadAndSplit::test_feature_count PASSED          [ 17%]
tests/test_data.py::TestLoadAndSplit::test_no_nulls_in_features PASSED   [ 23%]
tests/test_data.py::TestLoadAndSplit::test_no_nulls_in_labels PASSED     [ 29%]
tests/test_data.py::TestLoadAndSplit::test_label_classes PASSED          [ 35%]
tests/test_data.py::TestLoadAndSplit::test_stratification PASSED         [ 41%]
tests/test_data.py::TestLoadAndSplit::test_reproducibility PASSED        [ 47%]
tests/test_data.py::TestLoadAndSplit::test_feature_names_returned PASSED [ 52%]
tests/test_data.py::TestValidateData::test_valid_data_passes PASSED      [ 58%]
tests/test_data.py::Test